# Regulatory capital: FRTB SA and SA-CCR

This notebook demonstrates two standardized regulatory capital calculations exposed by `finstack_quant.margin`, plus initial-margin methodology selection:

- **FRTB SBA** (Fundamental Review of the Trading Book, Sensitivities-Based Approach) -- market-risk capital from delta / vega / curvature sensitivities across risk classes, with default risk charge (DRC) and the residual-risk add-on (RRAO).
- **SA-CCR** (Standardised Approach for Counterparty Credit Risk) -- the exposure-at-default (EAD) of a derivatives netting set as `alpha * (RC + PFE)`.
- **Initial margin** -- selecting an `ImMethodology` (SIMM, schedule, haircut, ...).

All inputs and results are plain numbers / dictionaries, so they cross the Python / Rust boundary unchanged.

## Imports

In [1]:
import pandas as pd

from finstack_quant.margin import (
    FrtbSensitivities,
    FrtbSbaEngine,
    frtb_sba_charge,
    SaCcrTrade,
    SaCcrNettingSetConfig,
    SaCcrEngine,
    saccr_ead,
    ImMethodology,
)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## FRTB SBA: building a sensitivity set

`FrtbSensitivities` collects standardized sensitivities by risk class. Conventions:

- **GIRR delta** is reported as base-currency P&L per a **1 percentage-point** parallel rate move at each canonical tenor (roughly `100 x DV01`). Tenor labels are case-sensitive (`"2Y"`, `"5Y"`, `"10Y"`, ...).
- **Vega** is supplied per option maturity / underlying tenor; **curvature** as the up / down CVR shocks.
- **RRAO** notional carries a 1% weight for exotic instruments, 0.1% otherwise.

Below is a rates-led trading book with some credit, equity, and FX delta.

In [2]:
sens = FrtbSensitivities("USD")

# GIRR (interest-rate) delta: base-ccy P&L per 1pp parallel shift at each tenor
sens.add_girr_delta("2Y", 12_000.0)
sens.add_girr_delta("5Y", 25_000.0)
sens.add_girr_delta("10Y", -8_000.0)

# Credit-spread (non-securitisation) delta
sens.add_csr_delta("ACME_CORP", 1, "5Y", 4_000.0)

# Equity delta (single name, bucket 1) and FX delta vs reporting currency
sens.add_equity_delta("ACME_CORP", 1, 10_000.0)
sens.add_fx_delta("EUR", "USD", 15_000.0)

# GIRR vega and curvature
sens.add_girr_vega("1Y", "5Y", 5_000.0)
sens.add_girr_curvature(2_000.0, -1_800.0)

# Residual-risk add-on for an exotic
sens.add_rrao_position("EXOTIC_TARN", 1_000_000.0, is_exotic=True)

sens.base_currency

'USD'

## FRTB SBA: the capital charge

`frtb_sba_charge` returns `(total_charge, breakdown)`. With no fixed correlation scenario it evaluates the low / medium / high correlation scenarios and reports the **binding** (maximum) one, then adds DRC and RRAO on top.

In [3]:
total, breakdown = frtb_sba_charge(sens)

print(f"Total SBA charge: {total:,.0f} {sens.base_currency}")
print(f"Binding correlation scenario: {breakdown['binding_scenario']}")
print(f"DRC: {breakdown['drc']:,.0f}   RRAO: {breakdown['rrao']:,.0f}")

Total SBA charge: 828,521 USD
Binding correlation scenario: low
DRC: 0   RRAO: 10,000


In [4]:
rows = []
for component in ("delta", "vega", "curvature"):
    for risk_class, charge in breakdown[component].items():
        rows.append({"component": component, "risk_class": risk_class, "charge": charge})

charge_df = pd.DataFrame(rows).sort_values(["component", "risk_class"]).reset_index(drop=True)
charge_df

,component,risk_class,charge
0,curvature,GIRR,"2,000.00"
1,delta,CSR_NON_SEC,"2,000.00"
2,delta,EQUITY,"550,000.00"
3,delta,FX,"225,000.00"
4,delta,GIRR,"34,520.79"
5,vega,GIRR,"5,000.00"


### Correlation scenarios

The SBA recomputes aggregation under three prescribed correlation levels. The binding scenario drives the capital number; DRC and RRAO are then added on top.

In [5]:
scenario_charges = breakdown["scenario_charges"]
binding = breakdown["binding_scenario"]

scen_df = pd.DataFrame(
    [{"scenario": s, "sba_charge": scenario_charges[s], "binding": s == binding} for s in ("low", "medium", "high")]
)

reconstructed = scenario_charges[binding] + breakdown["drc"] + breakdown["rrao"]
print(f"binding SBA + DRC + RRAO = {reconstructed:,.0f}  (matches total: {abs(reconstructed - total) < 1e-6})")
scen_df

binding SBA + DRC + RRAO = 828,521  (matches total: True)


,scenario,sba_charge,binding
0,low,"818,520.79",True
1,medium,"818,410.57",False
2,high,"818,300.00",False


In [6]:
# Object-oriented entry point; pin a single correlation scenario explicitly
high_total, _ = FrtbSbaEngine("high").calculate(sens)
print(f"FrtbSbaEngine('high') total: {high_total:,.0f}")

FrtbSbaEngine('high') total: 828,300


## SA-CCR: trades and netting set

`SaCcrTrade` describes a single derivative (asset class, notional, start / end date, underlier, hedging set, direction, current mark). A netting set is configured as either **unmargined** or **margined** (with threshold, MTA, NICA, and an MPOR in days).

In [7]:
ir_swap = SaCcrTrade(
    "IRS_5Y", "ir", 100_000_000.0,
    2024, 1, 15, 2029, 1, 15,
    "USD", "USD_IRS",
    direction=1.0, mtm=2_500_000.0,
)
fx_fwd = SaCcrTrade(
    "FXFWD_EURUSD", "fx", 50_000_000.0,
    2024, 1, 15, 2025, 7, 15,
    "EURUSD", "EUR_USD",
    direction=-1.0, mtm=-500_000.0,
)
trades = [ir_swap, fx_fwd]
[(t.trade_id, t.asset_class, f"{t.notional:,.0f}", f"{t.mtm:,.0f}") for t in trades]

[('IRS_5Y', 'INTEREST_RATE', '100,000,000', '2,500,000'),
 ('FXFWD_EURUSD', 'FOREIGN_EXCHANGE', '50,000,000', '-500,000')]

## SA-CCR: EAD for an unmargined netting set

`SaCcrEngine.calculate_ead` returns replacement cost (RC), potential future exposure (PFE), the PFE multiplier, the aggregate add-on (and its breakdown by asset class), the supervisory `alpha`, and `EAD = alpha * (RC + PFE)`.

In [8]:
unmargined = SaCcrNettingSetConfig.unmargined("MEGABANK", "CSA_UNCOLL", 0.0, 2024, 1, 15)
res_u = SaCcrEngine().calculate_ead(unmargined, trades)

print(f"alpha = {res_u['alpha']:.2f}   multiplier = {res_u['multiplier']:.4f}")
print(f"RC = {res_u['rc']:,.0f}   PFE = {res_u['pfe']:,.0f}   EAD = {res_u['ead']:,.0f}")
pd.DataFrame(sorted(res_u["add_on_by_asset_class"].items()), columns=["asset_class", "add_on"])

alpha = 1.40   multiplier = 1.0000
RC = 2,000,000   PFE = 4,214,126   EAD = 8,699,776


,asset_class,add_on
0,FOREIGN_EXCHANGE,"2,000,000.00"
1,INTEREST_RATE,"2,214,125.58"


## SA-CCR: margining reduces EAD

A CSA with collateral, a threshold, MTA, and a margin period of risk (MPOR) lowers both RC (collateral offsets exposure) and PFE (only the MPOR window is at risk).

In [9]:
margined = SaCcrNettingSetConfig.margined(
    "MEGABANK", "CSA_COLL",
    1_000_000.0,   # collateral held
    500_000.0,     # threshold
    250_000.0,     # minimum transfer amount
    0.0,           # net independent collateral amount
    10,            # MPOR (days)
    2024, 1, 15,
)
res_m = SaCcrEngine().calculate_ead(margined, trades)

pd.DataFrame([
    {"netting_set": "unmargined", "rc": res_u["rc"], "pfe": res_u["pfe"], "ead": res_u["ead"]},
    {"netting_set": "margined", "rc": res_m["rc"], "pfe": res_m["pfe"], "ead": res_m["ead"]},
])

,netting_set,rc,pfe,ead
0,unmargined,"2,000,000.00","4,214,125.58","8,699,775.81"
1,margined,"1,000,000.00","1,264,237.67","3,169,932.74"


In [10]:
# Convenience shortcut for a single bilateral set (alpha = 1.4, zero threshold)
rc, pfe, ead = saccr_ead(trades)
print(f"saccr_ead -> RC={rc:,.0f}  PFE={pfe:,.0f}  EAD={ead:,.0f}")

saccr_ead -> RC=2,000,000  PFE=4,214,126  EAD=8,699,776


## Initial-margin methodology selection

`ImMethodology` enumerates the supported IM regimes. (IM amounts -- `ImResult` -- are produced by the Rust calculators; the methodology selector is what is exposed to Python today.)

In [11]:
methods = ["haircut", "simm", "schedule", "internal_model", "clearing_house"]
pd.DataFrame([{"input": m, "methodology": str(ImMethodology.from_str(m))} for m in methods])

,input,methodology
0,haircut,haircut
1,simm,simm
2,schedule,schedule
3,internal_model,internal_model
4,clearing_house,clearing_house


## Takeaways

- `frtb_sba_charge(sensitivities)` returns `(total, breakdown)`; the total is the binding correlation scenario's SBA charge plus DRC and RRAO. `FrtbSbaEngine(scenario)` pins a single scenario.
- GIRR delta is base-currency P&L per 1pp move (~`100 x DV01`); pre-scale your sensitivities to the FRTB convention before loading them.
- `SaCcrEngine.calculate_ead(config, trades)` yields `RC`, `PFE`, the add-on breakdown, and `EAD = alpha * (RC + PFE)`; margined netting sets carry materially lower EAD than unmargined ones.
- `ImMethodology` selects the initial-margin regime (SIMM, schedule, haircut, internal model, clearing house).